# 💰 07 · 挖矿经济学 & 攻击模型

> 矿工不是慈善家。51% 攻击为什么没发生？

## 7.1 挖矿盈亏模型

```
日利润 = 日收入 - 日电费

日收入 = 区块奖励 × 日区块数 × 算力占比 × BTC 价格
日电费 = 矿机功耗(kW) × 24h × 电价($/kWh)
```

**关键变量**：BTC 价格、全网算力、电价、矿机能效。

In [ ]:
def mining_profitability(
    btc_price: float, network_hashrate: float,
    your_hashrate: float, power_w: float,
    electricity_cost: float, block_reward: float = 3.125
):
    # Daily revenue
    blocks_per_day = 144
    your_share = your_hashrate / network_hashrate
    daily_btc = blocks_per_day * block_reward * your_share
    daily_usd = daily_btc * btc_price
    
    # Daily cost
    daily_kwh = (power_w / 1000) * 24
    daily_cost = daily_kwh * electricity_cost
    
    profit = daily_usd - daily_cost
    roi_pct = (profit / daily_cost * 100) if daily_cost > 0 else 0
    
    print(f"BTC Price: ${btc_price:,.0f}  |  Electricity: ${electricity_cost:.3f}/kWh")
    print(f"Your hashrate: {your_hashrate/1e12:.0f} TH/s  |  Network: {network_hashrate/1e18:.0f} EH/s")
    print(f"Your share: {your_share*100:.8f}%")
    print(f"Daily revenue: ${daily_usd:.2f}  |  Daily cost: ${daily_cost:.2f}")
    print(f"Daily profit:  ${profit:.2f}  |  ROI: {roi_pct:.1f}%")
    return profit

# Scenario: S21 miner, $0.05/kWh electricity
print("🟢 PROFITABLE (S21, cheap power)")
mining_profitability(
    btc_price=100_000,
    network_hashrate=600e18,  # 600 EH/s
    your_hashrate=200e12,     # 200 TH/s
    power_w=3500,
    electricity_cost=0.05,
)

print("
🔴 UNPROFITABLE (S21, expensive power)")
mining_profitability(
    btc_price=70_000,
    network_hashrate=600e18,
    your_hashrate=200e12,
    power_w=3500,
    electricity_cost=0.15,
)

print("
💀 YOUR MACBOOK")
mining_profitability(
    btc_price=100_000,
    network_hashrate=600e18,
    your_hashrate=10_000,  # optimistic
    power_w=30,
    electricity_cost=0.10,
)

## 7.2 51% 攻击

**定义**：控制 >50% 全网算力的实体可以：
- ✅ **双花**：发交易 → 等确认 → 秘密挖更长的链 → 广播替代
- ✅ **交易审查**：拒绝打包特定交易
- ❌ **偷别人的币**：没有私钥做不到
- ❌ **凭空造币**：违反共识规则

**为什么还没发生？**
- 攻击 BTC 需要 300+ EH/s ASIC → 硬件成本 >$100 亿
- 攻击导致 BTC 崩盘 → 矿机变废铁
- **博弈论**：诚实挖矿长期收益 >> 攻击短期收益

In [ ]:
def simulate_double_spend_success(attacker_pct: float, confirmations: int = 6, trials: int = 5000):
    # Monte Carlo simulation of double-spend success probability
    successes = 0
    for _ in range(trials):
        honest = 0
        attacker = 0
        while honest < confirmations:
            if random.random() < attacker_pct:
                attacker += 1
            else:
                honest += 1
        
        # Attacker tries to catch up
        deficit = honest - attacker
        while deficit > 0:
            if random.random() < attacker_pct:
                attacker += 1
            else:
                honest += 1
            deficit = honest - attacker
        
        if deficit <= 0:
            successes += 1
    
    return successes / trials

print("Double-spend attack success rate (6 confirmations):")
print("-" * 45)
for pct in [0.10, 0.20, 0.30, 0.40, 0.45, 0.49, 0.51, 0.55]:
    rate = simulate_double_spend_success(pct, confirmations=6, trials=3000)
    bar = "█" * int(rate * 40)
    print(f"  {pct*100:5.1f}% hashrate: {rate*100:5.1f}% {bar}")

print("
💡 Even at 51%, success is ~70%—not 100%")
print("💡 Bitcoin's 6-confirmation rule provides strong probabilistic safety")

## 7.3 自私挖矿 (Selfish Mining)

**策略**：挖到块不广播，秘密继续挖。在适当时机释放。

- 理论上 <50% 算力也可能获利（Ittay Eyal, 2013）
- 实际上极难执行（需要精准网络时序控制）
- 以太坊的 Uncle Block 机制部分缓解了这个问题

**目前没有已知的自私挖矿攻击成功案例。**

## 7.4 MEV (Maximal Extractable Value)

MEV 是以太坊的概念，但原理普适：

矿工/验证者通过**重排、插入、审查**交易获利：
- 抢跑：看到大买单 → 自己先买 → 高价卖回
- 三明治攻击：在大单前后各插一个交易
- 清算套利

**MEV 暴露了一个深层问题**：即使共识安全，激励设计仍可能有漏洞。目前 Flashbots 等协议试图将 MEV 公开化、竞争化而非封闭化。

## 小结

✅ 挖矿是生意——盈亏取决于电价、币价、算力
✅ 51% 攻击在比特币上经济不可行（成本 >$100亿）
✅ 自私挖矿理论上存在，实际无人执行
✅ MEV 是新一代区块链经济学前沿课题
✅ **安全 = 密码学 + 博弈论 + 经济激励**